# Coastal Erosion Forecasting System
This notebook contains the full pipeline for processing coastal data, training the forecasting model, and visualizing future shoreline changes.

## Dependencies & Setup

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import xml.etree.ElementTree as ET
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from scipy.spatial.distance import cdist
import datetime
%matplotlib inline


## Data Loader (src/data_loader.py)

In [9]:
import os
import re
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime
import glob

def parse_kml_date(filename):
    """
    Extracts date from filename like 'SWnew MMDDYYYY.kml'.
    Example: 'SWnew 10222010.kml' -> datetime(2010, 10, 22)
    """
    match = re.search(r'(\d{1,2})(\d{2})(\d{4})', filename)
    if match:
        month, day, year = match.groups()
        return datetime(int(year), int(month), int(day))
    return None

def read_kml_shoreline(filepath):
    """
    Parses a KML file and returns a list of (lon, lat) tuples.
    """
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        coordinates = []
        for elem in root.iter():
            if 'coordinates' in elem.tag:
                text = elem.text.strip()
                coords_list = text.split()
                for coord in coords_list:
                    parts = coord.split(',')
                    if len(parts) >= 2:
                        lon = float(parts[0])
                        lat = float(parts[1])
                        coordinates.append([lon, lat])
                if coordinates:
                    break
        return coordinates
    except Exception as e:
        print(f"Error parsing {filepath}: {e}")
        return []

def load_all_shorelines(kml_dir):
    """
    Loads all KML files from the directory.
    Returns a dataframe with ['Date', 'Coordinates'].
    """
    files = glob.glob(os.path.join(kml_dir, "*.kml"))
    data = []
    
    for f in files:
        filename = os.path.basename(f)
        date_obj = parse_kml_date(filename)
        if date_obj:
            coords = read_kml_shoreline(f)
            if coords:
                data.append({'Date': date_obj, 'Coordinates': coords})
        else:
            # Try alternative pattern if needed, or just report
            # print(f"Skipping {filename}: Could not parse date")
            pass
            
    df = pd.DataFrame(data)
    if not df.empty:
        df = df.sort_values('Date').reset_index(drop=True)
    return df

def load_transect_stats(csv_path):
    """
    Loads the new transect-based statistics (CSV).
    """
    try:
        df = pd.read_csv(csv_path)
        return df
    except Exception as e:
        print(f"Error loading stats CSV: {e}")
        return pd.DataFrame()

# if __name__ == "__main__":


## Preprocessing (src/preprocessing.py)

In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import cdist

def create_baseline(all_coords):
    """
    Fits a straight line to all shoreline points to serve as a baseline.
    Returns (slope, intercept, min_lat, max_lat).
    Note: lat is Y, lon is X. We treat Longitude as function of Latitude for vertical-ish coastlines,
    or Latitude as function of Longitude for horizontal-ish.
    Kalmunai is roughly North-South, so we fit Lon = m * Lat + c
    """
    # Flatten all coordinates
    all_points = np.vstack(all_coords)
    X_lon = all_points[:, 0]
    y_lat = all_points[:, 1]
    
    # Check orientation: variance in Lat vs Lon
    # If var(Lat) > var(Lon), it's North-South (Kalmunai). fit Lon = f(Lat)
    if np.var(y_lat) > np.var(X_lon):
        # Fit Lon = m*Lat + c
        reg = LinearRegression().fit(y_lat.reshape(-1, 1), X_lon)
        slope = reg.coef_[0]
        intercept = reg.intercept_
        orientation = 'NS' # North-South
        min_dim = y_lat.min()
        max_dim = y_lat.max()
    else:
        # Fit Lat = m*Lon + c
        reg = LinearRegression().fit(X_lon.reshape(-1, 1), y_lat)
        slope = reg.coef_[0]
        intercept = reg.intercept_
        orientation = 'EW' # East-West
        min_dim = X_lon.min()
        max_dim = X_lon.begin() # Typo fix: max()
        max_dim = X_lon.max()

    return slope, intercept, orientation, min_dim, max_dim

def generate_transects(slope, intercept, orientation, min_dim, max_dim, num_transects=50):
    """
    Generates transect lines perpendicular to the baseline.
    Returns a list of transects, each defined by a point on the baseline and a direction vector.
    """
    # Generate points along the baseline
    dim_values = np.linspace(min_dim, max_dim, num_transects)
    
    transects = []
    
    if orientation == 'NS':
        # Baseline: Lon = slope * Lat + intercept
        # Vector along baseline: (slope, 1)  [dLon, dLat]
        # Normal vector (perpendicular): (1, -slope) or (-1, slope)
        
        # Let's normalize the normal vector
        normal_vec = np.array([1, -slope])
        normal_vec = normal_vec / np.linalg.norm(normal_vec)
        
        for lat in dim_values:
            lon_base = slope * lat + intercept
            origin = np.array([lon_base, lat])
            transects.append({'origin': origin, 'vector': normal_vec})
            
    else: # EW
        # Baseline: Lat = slope * Lon + intercept
        # Vector along baseline: (1, slope)
        # Normal vector: (-slope, 1)
        
        normal_vec = np.array([-slope, 1])
        normal_vec = normal_vec / np.linalg.norm(normal_vec)
        
        for lon in dim_values:
            lat_base = slope * lon + intercept
            origin = np.array([lon, lat_base])
            transects.append({'origin': origin, 'vector': normal_vec})
            
    return transects

def get_intersection_distance(transect, shoreline_coords):
    """
    Finds the intersection of a transect line with the shoreline.
    Returns resistance (distance) from baseline.
    Simple method: Find the shoreline point closest to the transect line.
    (Geometrically rigorous intersection is harder with discreet points, 
    nearest point is a good approximation if resolution is high).
    """
    origin = transect['origin']
    vec = transect['vector']
    
    # Project shoreline points onto the normal vector relative to origin
    # Projected distance = Dot(point - origin, vector)
    # We want the point that is ON the line defined by origin + t * vec
    # But actually, we just want the distance from origin to shoreline in the direction of 'vec'.
    # Since shoreline is rough, we find the point on shoreline that intersects the ray.
    
    # Alternative robust approach:
    # 1. Select shoreline points within a narrow band of the transect.
    # 2. Average their distances or take the closest.
    
    shoreline_arr = np.array(shoreline_coords)
    
    # Calculate distance of each shoreline point to the transect LINE (not ray)
    # Line defined by P = origin + t * vec
    # Distance to line = |det([vec, point-origin])| / |vec|  (2D cross product magnitude)
    # |vec| is 1.
    
    d_vecs = shoreline_arr - origin
    
    # Cross product in 2D: x1*y2 - x2*y1
    # vec = (vx, vy)
    # d_vec = (dx, dy)
    # cross = vx*dy - vy*dx
    cross_products = vec[0] * d_vecs[:, 1] - vec[1] * d_vecs[:, 0]
    
    # Filter points very close to the line (within some epsilon)
    # This simulates finding the intersection
    # Epsilon depends on data scale. decimal degrees. 1e-4 is ~10m.
    mask = np.abs(cross_products) < 1e-4 
    
    candidates = shoreline_arr[mask]
    
    if len(candidates) == 0:
        # Fallback: just find closest point absolutely? 
        # Or maybe widen search
        closest_idx = np.argmin(np.abs(cross_products))
        candidates = shoreline_arr[closest_idx:closest_idx+1]
        
    # Now find the distance along the vector for these candidates
    # Dot product
    candidate_d_vecs = candidates - origin
    distances = candidate_d_vecs[:, 0] * vec[0] + candidate_d_vecs[:, 1] * vec[1]
    
    # If multiple, take average position
    return np.mean(distances)

def process_shorelines(df_shorelines, num_transects=50):
    """
    Main processing function.
    1. Create baseline.
    2. Cast transects.
    3. Calculate distances for each date.
    """
    # 1. Fit global baseline
    all_coords = df_shorelines['Coordinates'].tolist()
    # Need to flatten logic slightly different than create_baseline expects if passed list of lists
    # create_baseline expects list of list of [x,y]
    slope, intercept, orient, min_d, max_d = create_baseline(all_coords)
    
    print(f"Baseline fitted. Orientation: {orient}, Slope: {slope:.4f}")
    
    # 2. Generate transects
    transects = generate_transects(slope, intercept, orient, min_d, max_d, num_transects)
    
    # 3. Calculate distances
    # Structure: Date, T1, T2, ... Tn
    results = []
    
    for _, row in df_shorelines.iterrows():
        date = row['Date']
        coords = row['Coordinates']
        
        row_data = {'Date': date}
        for i, tr in enumerate(transects):
            dist = get_intersection_distance(tr, coords)
            row_data[f'Transect_{i}'] = dist
            
        results.append(row_data)
        
    return pd.DataFrame(results)

# if __name__ == "__main__":


## Forecasting Engine (src/forecasting.py)

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import datetime

def prepare_features(df):
    """
    Converts Date to ordinal for regression.
    """
    if 'Date' in df.columns:
        df = df.sort_values('Date')
        # Use days since first observation as feature to keep numbers small
        base_date = df['Date'].min()
        df['Days'] = (df['Date'] - base_date).dt.days
        return df, base_date
    return df, None

def train_and_predict(df, transect_cols, years_ahead=5):
    """
    Trains a Random Forest for each transect and forecasts future position.
    """
    df_processed, base_date = prepare_features(df)
    
    # Define future target dates
    last_date = df_processed['Date'].max()
    future_dates = []
    # Forecast annually for 5 years
    for i in range(1, years_ahead + 1):
        future_dates.append(last_date + datetime.timedelta(days=365 * i))
    
    future_days = [(d - base_date).days for d in future_dates]
    X_future = np.array(future_days).reshape(-1, 1)
    
    X = df_processed[['Days']].values
    
    predictions = {}
    models = {}
    
    print(f"Training models for {len(transect_cols)} transects...")
    
    for col in transect_cols:
        y = df_processed[col].values
        # Drop NaNs if any
        mask = ~np.isnan(y)
        if np.sum(mask) < 2:
            predictions[col] = [np.nan] * len(future_dates)
            continue
            
        # Linear Regression for trend extrapolation
        # RF cannot extrapolate beyond training range
        model = LinearRegression()
        model.fit(X[mask], y[mask])
        
        # Predict
        preds = model.predict(X_future)
        predictions[col] = preds
        models[col] = model
        
    # Construct result DataFrame
    future_df = pd.DataFrame({'Date': future_dates})
    for col, preds in predictions.items():
        future_df[col] = preds
        
    return future_df, models

# if __name__ == "__main__":


## Visualization (src/visualization.py)

In [12]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_transect_history(history_df, future_df, transect_cols, output_path):
    """
    Plots the average erosion/accretion trend over time.
    """
    plt.figure(figsize=(10, 6))
    
    # Calculate average distance across all transects to show general trend
    avg_hist = history_df[transect_cols].mean(axis=1)
    avg_fut = future_df[transect_cols].mean(axis=1)
    
    plt.plot(history_df['Date'], avg_hist, 'o-', label='Historical (Avg)')
    plt.plot(future_df['Date'], avg_fut, 'x--', label='Forecast (Avg)')
    
    plt.xlabel('Year')
    plt.ylabel('Average Distance from Baseline (deg/m equivalent)')
    plt.title('Shoreline Position Forecast (Average across all transects)')
    plt.legend()
    plt.grid(True)
    plt.savefig(output_path)
    plt.close()

def reconstruct_shoreline(transects, distances):
    """
    Reconstructs (lon, lat) points from transect origin + vector * distance.
    """
    points = []
    for tr, dist in zip(transects, distances):
        if np.isnan(dist):
            continue
        # point = origin + vector * distance
        pt = tr['origin'] + tr['vector'] * dist
        points.append(pt)
    return np.array(points)

def plot_map_visualization(history_df, future_df, transect_config, output_path):
    """
    Plots the physical map view of shorelines (Historical vs Predicted).
    """
    # Recalculate transect geometry to map distances back to coordinates
    # We need the baseline parameters. 
    # Since we don't save them easily, we'll re-derive or pass them.
    # Actually, main.py has the transect objects if we reconstruct them.
    # To keep it loosely coupled, let's just assume we re-run generic baseline.
    pass 
    # Skipping complex map reconstruction in this simplified function 
    # unless we pass the transect objects explicitly.
    
    # Simplified: Just plot the raw points if available, but future only has distances.
    # We need the transect definitions to inversing.
    
    # Moving this logic to main.py where we have the transects.


## Interactive Forecast Logic

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

def interactive_main():
    print("--- Coastal Erosion Interactive Forecast ---")
    try:
        target_year = int(input("Enter the target year for forecast (e.g., 2028): "))
    except ValueError:
        print("Invalid year. Please enter a number.")
        return

    base_dir = r"e:\coastal\data"
    kml_dir = os.path.join(base_dir, "high_res_kml")
    output_dir = r"e:\coastal\output"
    
    print("1. Loading Data & Training Model...")
    shorelines_df = load_all_shorelines(kml_dir)
    
    # Process Transects
    all_coords = shorelines_df['Coordinates'].tolist()
    slope, intercept, orient, min_d, max_d = create_baseline(all_coords)
    transects = generate_transects(slope, intercept, orient, min_d, max_d, num_transects=50)
    
    results = []
    for _, row in shorelines_df.iterrows():
        date = row['Date']
        coords = row['Coordinates']
        row_data = {'Date': date}
        for i, tr in enumerate(transects):
            dist = get_intersection_distance(tr, coords)
            row_data[f'Transect_{i}'] = dist
        results.append(row_data)
    
    history_df = pd.DataFrame(results).sort_values('Date')
    transect_cols = [c for c in history_df.columns if c.startswith('Transect_')]
    
    # Determine years ahead
    latest_date = history_df['Date'].max()
    years_ahead = target_year - latest_date.year
    
    if years_ahead <= 0:
        print(f"Target year {target_year} must be in the future (after {latest_date.year}).")
        return
        
    print(f"Forecasting {years_ahead} years ahead to {target_year}...")
    
    # Train & Predict
    # train_and_predict returns annual, we just need the row for target year
    # But train_and_predict logic loops. Let's reuse it but filter result.
    future_df, models = train_and_predict(history_df, transect_cols, years_ahead=years_ahead)
    
    # Create a proper future dataframe for plotting
    # We have future_df from train_and_predict which contains annual steps up to target
    # So we can pass that directly.
    
    # Extract Target Prediction
    final_forecast = future_df.iloc[-1]
    forecast_date = final_forecast['Date']
    print(f"Forecast Target Date: {forecast_date.date()}")
    
    # --- Generate Outputs ---
    
    # 0. Trend Plot
    print("Generating Trend Plot...")
    trend_file = os.path.join(output_dir, f"trend_forecast_{target_year}.png")
    plot_transect_history(history_df, future_df, transect_cols, trend_file)
    print(f"Trend Plot saved: {trend_file}")

    # 1. Map Overlay
    print("Generating Map...")
    latest_hist = history_df.iloc[-1]
    
    plt.figure(figsize=(12, 12))
    
    # Plot ALL historical shorelines (gray, faint)
    print("Plotting historical shorelines...")
    for _, row in history_df.iterrows():
        hist_pts_i = []
        for i, col in enumerate(transect_cols):
            d = row[col]
            if np.isnan(d): continue
            tr = transects[i]
            pt = tr['origin'] + tr['vector'] * d
            hist_pts_i.append(pt)
        if hist_pts_i:
            pts_arr = np.array(hist_pts_i)
            plt.plot(pts_arr[:,0], pts_arr[:,1], color='gray', alpha=0.3, linewidth=0.5)
    
    # Plot Latest Observed (Blue)
    hist_pts = []
    forecast_pts = []
    valid_indices = []
    
    for i, col in enumerate(transect_cols):
        idx = int(col.split('_')[1])
        tr = transects[idx]
        d_h = latest_hist[col]
        d_f = final_forecast[col]
        
        if np.isnan(d_h) or np.isnan(d_f):
            continue
            
        valid_indices.append(idx)
        p_h = tr['origin'] + tr['vector'] * d_h
        p_f = tr['origin'] + tr['vector'] * d_f
        hist_pts.append(p_h)
        forecast_pts.append(p_f)
        
    hist_pts = np.array(hist_pts)
    forecast_pts = np.array(forecast_pts)
    
    plt.plot(hist_pts[:,0], hist_pts[:,1], 'b-', linewidth=2, label=f'Latest Observed ({latest_hist["Date"].date()})')
    plt.plot(forecast_pts[:,0], forecast_pts[:,1], 'r--', linewidth=3, label=f'Forecast ({forecast_date.date()})')
    
    # Fill
    for i in range(len(hist_pts)-1):
        # Color based on change
        d_h = latest_hist[transect_cols[valid_indices[i]]]
        d_f = final_forecast[transect_cols[valid_indices[i]]]
        change = d_f - d_h
        color = 'lightgreen' if change > 0 else 'salmon'
        
        # Check against next point availability
        if i+1 < len(hist_pts):
            poly = np.array([hist_pts[i], hist_pts[i+1], forecast_pts[i+1], forecast_pts[i]])
            plt.fill(poly[:,0], poly[:,1], color=color, alpha=0.5)
        
    plt.title(f"Shoreline Forecast Map: {target_year}\n(Gray=History, Blue=Latest, Red=Forecast)")
    plt.axis('equal')
    plt.legend()
    map_file = os.path.join(output_dir, f"map_forecast_{target_year}.png")
    plt.savefig(map_file)
    plt.close()
    print(f"Map saved: {map_file}")
    
    # 2. Stats
    print("\n--- Statistics (vs 2025) ---")
    distances = final_forecast[transect_cols].values
    hist_distances = latest_hist[transect_cols].values
    
    change = distances - hist_distances
    avg_shift = np.nanmean(change) * 111000 # meters approx
    max_acc = np.nanmax(change) * 111000
    max_ero = np.nanmin(change) * 111000
    
    print(f"Average Shift: {avg_shift:+.2f} meters")
    print(f"Max Accretion: {max_acc:+.2f} meters")
    print(f"Max Erosion:   {max_ero:+.2f} meters")
    
    # 3. KML Generation
    print("Generating KML...")
    kml_points = []
    # Use valid_indices or loop all to reconstruct
    # We need to loop all transects to get the full line strip
    for i, col in enumerate(transect_cols):
        idx = int(col.split('_')[1])
        tr = transects[idx]
        dist = final_forecast[col]
        
        if np.isnan(dist):
            continue
            
        pt = tr['origin'] + tr['vector'] * dist
        kml_points.append(pt)
        
    coord_str_list = []
    for p in kml_points:
        coord_str_list.append(f"{p[0]:.14f},{p[1]:.14f},0")
    
    coord_str = " ".join(coord_str_list)
    
    kml_content = f"""<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
<Document>
	<name>Forecast {target_year}</name>
	<Style id="red_line">
		<LineStyle>
			<color>ff0000ff</color>
			<width>4</width>
		</LineStyle>
	</Style>
	<Placemark>
		<name>Forecast Shoreline {target_year}</name>
		<styleUrl>#red_line</styleUrl>
		<LineString>
			<tessellate>1</tessellate>
			<coordinates>
				{coord_str}
			</coordinates>
		</LineString>
	</Placemark>
</Document>
</kml>
"""
    kml_file = os.path.join(output_dir, f"forecast_{target_year}.kml")
    with open(kml_file, "w") as f:
        f.write(kml_content)
    print(f"KML saved: {kml_file}")

    # Save CSV result
    csv_file = os.path.join(output_dir, f"forecast_data_{target_year}.csv")
    final_forecast.to_frame().T.to_csv(csv_file, index=False)
    print(f"Data saved: {csv_file}")
    
    input("\nPress Enter to exit...")

# if __name__ == "__main__":
#

## Run Forecast
Execute the cell below to start the interactive forecast.

In [15]:
# To run the interactive tool, uncomment the line below:
interactive_main()

--- Coastal Erosion Interactive Forecast ---
1. Loading Data & Training Model...


TypeError: 'module' object is not callable